In [6]:
import pandas as pd
import numpy as np


In [15]:
# 原始测量值
plate = pd.DataFrame({"T0": [0.751, 0.749, 0.751]}, index=["1", "2", "3"])
plastic = pd.DataFrame(
    {
        "D1": [100.26, 100.26, 100.28],
        "T1": [1.253, 1.254, 1.252],
    },
    index=["1", "2", "3"],
)
metal = pd.DataFrame(
    {
        "Do": [99.94, 100.00, 99.98],
        "Di": [93.98, 94.04, 94.00],
        "T2": [1.552, 1.553, 1.550],
    },
    index=["1", "2", "3"],
)
stick = pd.DataFrame({"l": [610.0, 610.0, 610.0], "T4": [2.145, 2.145, 2.145]}, index=["1", "2", "3"])

# 1) 质量表（先给占位，后续填入实测质量 g）
df_mass = pd.DataFrame(
    {
        "质量(kg)": [np.nan, 0.71571, 0.70045, 0.13141],
    },
    index=["金属载物盘", "塑料圆柱", "金属圆筒", "金属细杆"],
)

# 2) 几何尺寸：测量值 + 平均值
df_geom_meas = pd.DataFrame(
    {
        "塑料圆柱_D1(mm)": plastic["D1"],
        "金属圆筒_Do(mm)": metal["Do"],
        "金属圆筒_Di(mm)": metal["Di"],
        "金属细杆_l(mm)": stick["l"],
    }
)
df_geom_mean = pd.DataFrame(df_geom_meas.mean(axis=0), columns=["平均值"])

# 3) 周期：测量值 + 平均值
df_period_meas = pd.DataFrame(
    {
        "金属载物盘_T0(s)": plate["T0"],
        "塑料圆柱_T1(s)": plastic["T1"],
        "金属圆筒_T2(s)": metal["T2"],
        "金属细杆_T4(s)": stick["T4"],
    }
)
df_period_mean = pd.DataFrame(df_period_meas.mean(axis=0), columns=["平均值"])

# 4) I 表（理论值、实验值、Er），全部由上表计算
# 说明：
# - 理论值使用常见转动惯量公式（直径/长度均从 mm 转 m）
# - 实验值需要扭转常量 K；若没给 K，则保持 NaN

m_kg = df_mass["质量(kg)"]
geo_mean = df_geom_mean["平均值"] / 1000.0
period_mean = df_period_mean["平均值"]

I_theory = pd.Series(
    {
        "塑料圆柱": 0.125 * m_kg["塑料圆柱"] * geo_mean["塑料圆柱_D1(mm)"] ** 2,
        "金属圆筒": 0.125 * m_kg["金属圆筒"] * (geo_mean["金属圆筒_Do(mm)"] ** 2 + geo_mean["金属圆筒_Di(mm)"] ** 2),
        "金属细杆": (1.0 / 12.0) * m_kg["金属细杆"] * geo_mean["金属细杆_l(mm)"] ** 2,
    }
)
constants = {
    # 扭转常量，单位 kg*m^2/s^2
    "K": 4 * np.pi * I_theory["塑料圆柱"] / (period_mean["塑料圆柱_T1(s)"] ** 2 - period_mean["金属载物盘_T0(s)"] ** 2),
}
I0 = I_theory["塑料圆柱"] * period_mean["金属载物盘_T0(s)"] ** 2 / (period_mean["塑料圆柱_T1(s)"] ** 2 - period_mean["金属载物盘_T0(s)"] ** 2)
I4支 = 2.32 * 1e-5
I_exp = pd.Series(
    {
        "金属载物盘": I0,
        "塑料圆柱": constants["K"] * (period_mean["塑料圆柱_T1(s)"] ** 2) / (4 * np.pi) - I0,
        "金属圆筒": constants["K"] * (period_mean["金属圆筒_T2(s)"] ** 2) / (4 * np.pi) - I0,
        "金属细杆": constants["K"] * (period_mean["金属细杆_T4(s)"] ** 2) / (4 * np.pi) - I4支,
    }
)

df_I = pd.DataFrame(
    {
        "I理论值(kg*m^2)": I_theory,
        "I实验值(kg*m^2)": I_exp,
    }
)
df_I["Er(%)"] = np.abs(df_I["I实验值(kg*m^2)"] - df_I["I理论值(kg*m^2)"]) / df_I["I理论值(kg*m^2)"] * 100
df_I["Er(%)"] = df_I["Er(%)"].round(2)

df_mass, df_geom_meas, df_geom_mean, df_period_meas, df_period_mean, df_I


(        质量(kg)
 金属载物盘      NaN
 塑料圆柱   0.71571
 金属圆筒   0.70045
 金属细杆   0.13141,
    塑料圆柱_D1(mm)  金属圆筒_Do(mm)  金属圆筒_Di(mm)  金属细杆_l(mm)
 1       100.26        99.94        93.98       610.0
 2       100.26       100.00        94.04       610.0
 3       100.28        99.98        94.00       610.0,
                     平均值
 塑料圆柱_D1(mm)  100.266667
 金属圆筒_Do(mm)   99.973333
 金属圆筒_Di(mm)   94.006667
 金属细杆_l(mm)   610.000000,
    金属载物盘_T0(s)  塑料圆柱_T1(s)  金属圆筒_T2(s)  金属细杆_T4(s)
 1        0.751       1.253       1.552       2.145
 2        0.749       1.254       1.553       2.145
 3        0.751       1.252       1.550       2.145,
                   平均值
 金属载物盘_T0(s)  0.750333
 塑料圆柱_T1(s)   1.253000
 金属圆筒_T2(s)   1.551667
 金属细杆_T4(s)   2.145000,
        I理论值(kg*m^2)  I实验值(kg*m^2)  Er(%)
 塑料圆柱       0.000899      0.000899   0.00
 金属圆筒       0.001649      0.001648   0.08
 金属细杆       0.004075      0.004086   0.28
 金属载物盘           NaN      0.000503    NaN)